# CH06: PSI drift, monthly PR-AUC and retrain trigger (Olist)

Attach the Kaggle dataset `olistbr/brazilian-ecommerce` to this notebook and enable internet access. The data is read from `/kaggle/input/brazilian-ecommerce/`. The notebook builds monthly snapshots with the CH02 package, freezes a LightGBM model trained on snapshots up to 2017-10-01, scores every monthly snapshot, and runs the monitor. Run all cells top to bottom.

This notebook has not been executed on this machine, because the data is not here.

In [ ]:
!git clone -q https://github.com/Patotricks15/churn-gbm-classifier.git /kaggle/working/churn-gbm-classifier
!git clone -q https://github.com/Patotricks15/churn-drift-monitoring.git /kaggle/working/churn-drift-monitoring
!pip install -q -e /kaggle/working/churn-gbm-classifier
!pip install -q -e "/kaggle/working/churn-drift-monitoring[dev]"

In [ ]:
import pandas as pd
from lightgbm import LGBMClassifier

from churn.data import load_olist
from churn.features import FEATURE_COLUMNS
from churn.models import scale_pos_weight
from churn.pipeline import build_panel
from monitor.config import H_DAYS, REFERENCE_END

tx = load_olist("/kaggle/input/brazilian-ecommerce")
snapshots = pd.date_range("2017-04-01", "2018-05-01", freq="MS").strftime("%Y-%m-%d").tolist()
panel = build_panel(tx, snapshots, H_DAYS)
panel = panel[~panel["single_order"]].reset_index(drop=True)

train = panel[panel["snapshot_date"] <= REFERENCE_END]
model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=50,
    scale_pos_weight=scale_pos_weight(train["churned"]),
    random_state=42,
    n_jobs=1,
    verbose=-1,
)
model.fit(train[FEATURE_COLUMNS], train["churned"])

scores = panel[["snapshot_date", "churned", *FEATURE_COLUMNS]].copy()
scores["proba"] = model.predict_proba(panel[FEATURE_COLUMNS])[:, 1]
scores.to_csv("/kaggle/working/ch06_monthly_scores.csv", index=False)
print(scores.groupby("snapshot_date").size())

In [ ]:
!python /kaggle/working/churn-drift-monitoring/scripts/run_monitor.py --scores /kaggle/working/ch06_monthly_scores.csv --out-dir /kaggle/working/ch06_reports

In [ ]:
from IPython.display import Image, display

display(Image("/kaggle/working/ch06_reports/figures/monitor_psi_pr_auc.png"))